In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GroupShuffleSplit,
    GroupKFold,
    GridSearchCV,
    cross_validate,
)
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix as cm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn import svm as svm
from sklearn.ensemble import RandomForestClassifier


## Loading data

In [2]:
def _to_array(raw_features):
    """
    Convert stored feature payload to a 3D array: (windows, channels, n_features).
    Handles the dict stored in features.npz.
    """
    arr = np.asarray(raw_features)
    if arr.dtype == object and arr.shape == ():  # dict wrapped in 0-d object
        obj = arr.item()
        if isinstance(obj, dict):
            keys = sorted(obj.keys())  # consistent order
            stacked = np.stack([np.asarray(obj[k]) for k in keys], axis=-1)
            return stacked, keys
        return np.asarray(obj), None
    return arr, None

In [3]:
def load_feature_files(root="data", pattern="*_features.npz", target_channels=None):
      """Load features and standardize channel count by padding/truncating.
      If target_channels is None, use the max channel count found (pads smaller files)."""
      root = Path(root)
      collected = []
      for fp in sorted(root.rglob(pattern)):
          data = np.load(fp, allow_pickle=True)
          X_raw = data["features"]
          X, feat_keys = _to_array(X_raw)
          labels = data.get("window_labels")
          labels = np.array([None] * len(X), dtype=object) if labels is None else np.asarray(labels, dtype=object)
          collected.append((fp, X, labels, feat_keys, data))
      if not collected:
          raise FileNotFoundError(f"No feature files found under {root}")

      if target_channels is None:
          target_channels = max(item[1].shape[1] for item in collected)

      X_list, y_list, meta = [], [], []
      for fp, X, labels, feat_keys, data in collected:
          channels = X.shape[1]
          if channels < target_channels:
              pad = np.zeros((X.shape[0], target_channels - channels, X.shape[2]), dtype=X.dtype)
              X = np.concatenate([X, pad], axis=1)
          elif channels > target_channels:
              X = X[:, :target_channels, :]
          X_list.append(X)
          y_list.append(labels)
          meta.append({
              "path": str(fp),
              "feature_list": feat_keys or list(data["feature_list"]),
              "fs": float(np.asarray(data["fs"]).squeeze()),
              "window_size_samples": int(data["window_size_samples"]),
              "window_step_samples": int(data["window_step_samples"]),
              "window_start_samples": np.asarray(data["window_start_samples"]),
              "window_end_samples": np.asarray(data["window_end_samples"]),
              "window_label_confidence": data.get("window_label_confidence"),
          })
      return np.vstack(X_list), np.concatenate(y_list), meta

In [4]:
def load_features_by_label(root="data", pattern="*_features.npz", target_channels=None):
      X, y, meta = load_feature_files(root, pattern, target_channels=target_channels)
      mask = y != None  # noqa: E711
      return X[mask], y[mask], meta

def load_features_with_groups(root="data", pattern="*_features.npz", target_channels=None):
    X, y, meta = load_feature_files(root, pattern, target_channels=target_channels)
    groups = []
    for entry in meta:
        count = len(entry["window_start_samples"])
        groups.extend([entry["path"]] * count)
    groups = np.asarray(groups, dtype=object)
    mask = y != None  # noqa: E711
    return X[mask], y[mask], groups[mask]


### Classifiers

#### split data

In [5]:
# Load and split data
X_raw, y, meta = load_features_by_label(target_channels=None)  # auto-pads to max channel count
X = X_raw.reshape(X_raw.shape[0], -1)  # flatten channels x features

X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, stratify=y
  )

# Eval helper: 10-fold CV + holdout


def eval_model(name, model, X, y):
    cv = StratifiedKFold(n_splits=10, shuffle=True)
    cv_scores = cross_validate(model, X, y, cv=cv, scoring='accuracy', return_train_score=False)
    print(f"{name} 10-fold CV accuracy: {cv_scores['test_score'].mean():.3f} +/- {cv_scores['test_score'].std():.3f}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name} holdout accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print('Confusion matrix:\n', cm(y_test, y_pred))
    print('\nReport:\n', classification_report(y_test, y_pred))

### LDA

In [18]:
# LDA
lda = LDA(store_covariance=True)
eval_model('LDA', lda, X, y)

LDA 10-fold CV accuracy: 0.727 +/- 0.003
LDA holdout accuracy: 0.726
Confusion matrix:
 [[ 968   26   69    4   37   54]
 [  44  777  230    7   34   69]
 [  76   51 1435  103  204  155]
 [   2   37  110  946   16   50]
 [   2   36  214   15  753  140]
 [   2   13  191   50  101  804]]

Report:
               precision    recall  f1-score   support

        horn       0.88      0.84      0.86      1158
   left_turn       0.83      0.67      0.74      1161
     neutral       0.64      0.71      0.67      2024
  right_turn       0.84      0.81      0.83      1161
 signal_left       0.66      0.65      0.65      1160
signal_right       0.63      0.69      0.66      1161

    accuracy                           0.73      7825
   macro avg       0.75      0.73      0.74      7825
weighted avg       0.73      0.73      0.73      7825



### QDA

In [19]:
# QDA
qda = QDA(reg_param=0.5)
eval_model('QDA', qda, X, y)


QDA 10-fold CV accuracy: 0.744 +/- 0.007
QDA holdout accuracy: 0.748
Confusion matrix:
 [[1000   16   63   12   11   56]
 [  50  889  177    8   11   26]
 [  77   74 1446  134  172  121]
 [   1   39  118  968    9   26]
 [   8   41  255   42  681  133]
 [   3   23  159   61   47  868]]

Report:
               precision    recall  f1-score   support

        horn       0.88      0.86      0.87      1158
   left_turn       0.82      0.77      0.79      1161
     neutral       0.65      0.71      0.68      2024
  right_turn       0.79      0.83      0.81      1161
 signal_left       0.73      0.59      0.65      1160
signal_right       0.71      0.75      0.73      1161

    accuracy                           0.75      7825
   macro avg       0.76      0.75      0.76      7825
weighted avg       0.75      0.75      0.75      7825



### SVM

In [20]:
# SVM (scale features)
svm_clf = make_pipeline(StandardScaler(), svm.SVC(kernel='rbf', C=100, gamma='scale'))
eval_model('SVM (RBF)', svm_clf, X, y)

KeyboardInterrupt: 

### Random Forest

In [7]:
# Random Forest
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
eval_model('Random Forest', rf_clf, X, y)

Random Forest 10-fold CV accuracy: 0.902 +/- 0.002
Random Forest holdout accuracy: 0.898
Confusion matrix:
 [[2847    8  210    2   27   22]
 [  30 2867  180    3   17   19]
 [ 207  227 5160  244  209  223]
 [   4    9   90 3006    1    7]
 [  45   29  142    0 2863   37]
 [  26    8  135    8   53 2886]]

Report:
               precision    recall  f1-score   support

        horn       0.90      0.91      0.91      3116
   left_turn       0.91      0.92      0.92      3116
     neutral       0.87      0.82      0.85      6270
  right_turn       0.92      0.96      0.94      3117
 signal_left       0.90      0.92      0.91      3116
signal_right       0.90      0.93      0.91      3116

    accuracy                           0.90     21851
   macro avg       0.90      0.91      0.91     21851
weighted avg       0.90      0.90      0.90     21851



### Model Optimization (Group-aware)


In [ ]:
# Group-aware train/test split for optimization and NN tuning
X_raw_opt, y_opt, groups_opt = load_features_with_groups(target_channels=None)
X_opt = X_raw_opt.reshape(X_raw_opt.shape[0], -1)

unique_groups = np.unique(groups_opt)
if unique_groups.size >= 2:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(X_opt, y_opt, groups_opt))
    print(f"Using group split across {unique_groups.size} sessions/files.")
else:
    train_idx, test_idx = train_test_split(
        np.arange(len(y_opt)),
        test_size=0.2,
        stratify=y_opt,
        random_state=42,
    )
    print("Warning: group split disabled (not enough sessions/files).")

X_train_opt, X_test_opt = X_opt[train_idx], X_opt[test_idx]
y_train_opt, y_test_opt = y_opt[train_idx], y_opt[test_idx]
groups_train_opt = groups_opt[train_idx]
train_group_count = np.unique(groups_train_opt).size

if unique_groups.size >= 2 and train_group_count >= 2:
    n_splits = min(5, train_group_count)
    cv = GroupKFold(n_splits=n_splits)
    use_groups = True
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    use_groups = False
    if unique_groups.size >= 2 and train_group_count < 2:
        print("Warning: group CV disabled (not enough training groups).")


Using group split across 13 sessions/files.


In [ ]:
results = []


def run_grid(name, estimator, param_grid, scale=True):
    model = make_pipeline(StandardScaler(), estimator) if scale else estimator

    grid = GridSearchCV(
        model,
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    if use_groups:
        grid.fit(X_train_opt, y_train_opt, groups=groups_train_opt)
    else:
        grid.fit(X_train_opt, y_train_opt)

    best_model = grid.best_estimator_
    test_acc = float(best_model.score(X_test_opt, y_test_opt))
    results.append(
        {
            "model": name,
            "cv_best": float(grid.best_score_),
            "test_acc": test_acc,
            "best_params": grid.best_params_,
        }
    )
    print(f"{name}: cv={grid.best_score_:.3f} test={test_acc:.3f}")
    return best_model


# LDA
run_grid(
    "LDA",
    LDA(),
    [
        {"lineardiscriminantanalysis__solver": ["svd"]},
        {
            "lineardiscriminantanalysis__solver": ["lsqr", "eigen"],
            "lineardiscriminantanalysis__shrinkage": ["auto", 0.1, 0.3, 0.5, 0.7, 0.9],
        },
    ],
)

# QDA
run_grid(
    "QDA",
    QDA(),
    {"quadraticdiscriminantanalysis__reg_param": [0.3, 0.5, 0.7, 0.9]},
)

# SVM
run_grid(
    "SVM",
    svm.SVC(kernel="rbf"),
    {
        "svc__C": [0.1, 1, 10, 100, 1000],
        "svc__gamma": ["scale", "auto", 1e-4, 1e-3, 1e-2, 1e-1],
    },
)

# Random Forest
run_grid(
    "RandomForest",
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {
        "randomforestclassifier__n_estimators": [200, 400],
        "randomforestclassifier__max_depth": [None, 10, 20],
        "randomforestclassifier__min_samples_leaf": [1, 2, 4],
        "randomforestclassifier__class_weight": ["balanced", None],
    },
    scale=False,
)

results = sorted(results, key=lambda x: x["test_acc"], reverse=True)
print("\nSummary (sorted by test accuracy):")
for res in results:
    print(f"{res['model']}: test={res['test_acc']:.3f}, cv={res['cv_best']:.3f}, params={res['best_params']}")


LDA: cv=0.671 test=0.645
QDA: cv=0.617 test=0.598
SVM: cv=0.700 test=0.680


ValueError: Invalid parameter 'randomforestclassifier' for estimator RandomForestClassifier(n_jobs=-1, random_state=42). Valid parameters are: ['bootstrap', 'ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'max_samples', 'min_impurity_decrease', 'min_samples_leaf', 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'n_estimators', 'n_jobs', 'oob_score', 'random_state', 'verbose', 'warm_start'].

### Neural Network (PyTorch)


In [ ]:
np.random.seed(42)
torch.manual_seed(42)

if "X_train_opt" in globals():
    X_train_nn = X_train_opt.copy()
    y_train_nn = y_train_opt.copy()
    X_test_nn = X_test_opt.copy()
    y_test_nn = y_test_opt.copy()
    groups_train_nn = groups_train_opt.copy()
    use_groups_nn = bool(use_groups)
else:
    X_raw_nn, y_nn, groups_nn = load_features_with_groups(target_channels=None)
    X_nn = X_raw_nn.reshape(X_raw_nn.shape[0], -1)
    unique_groups = np.unique(groups_nn)
    if unique_groups.size >= 2:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(splitter.split(X_nn, y_nn, groups_nn))
        print(f"Using group split across {unique_groups.size} sessions/files.")
    else:
        train_idx, test_idx = train_test_split(
            np.arange(len(y_nn)),
            test_size=0.2,
            stratify=y_nn,
            random_state=42,
        )
        print("Warning: group split disabled (not enough sessions/files).")

    X_train_nn, X_test_nn = X_nn[train_idx], X_nn[test_idx]
    y_train_nn, y_test_nn = y_nn[train_idx], y_nn[test_idx]
    groups_train_nn = groups_nn[train_idx]
    use_groups_nn = unique_groups.size >= 2

# Train/val split
if use_groups_nn and np.unique(groups_train_nn).size >= 2:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=123)
    tr_idx, val_idx = next(splitter.split(X_train_nn, y_train_nn, groups_train_nn))
else:
    perm = np.random.RandomState(123).permutation(len(y_train_nn))
    val_size = int(len(y_train_nn) * 0.2)
    val_idx = perm[:val_size]
    tr_idx = perm[val_size:]

X_tr, X_val = X_train_nn[tr_idx], X_train_nn[val_idx]
y_tr, y_val = y_train_nn[tr_idx], y_train_nn[val_idx]

mean = X_tr.mean(axis=0, keepdims=True)
std = X_tr.std(axis=0, keepdims=True)
std[std == 0] = 1.0
X_tr = (X_tr - mean) / std
X_val = (X_val - mean) / std
X_test_nn = (X_test_nn - mean) / std

le = LabelEncoder()
y_tr_enc = le.fit_transform(y_tr)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test_nn)

train_ds = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr_enc, dtype=torch.long),
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val_enc, dtype=torch.long),
)
test_ds = TensorDataset(
    torch.tensor(X_test_nn, dtype=torch.float32),
    torch.tensor(y_test_enc, dtype=torch.long),
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512)
test_loader = DataLoader(test_ds, batch_size=512)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
def make_mlp(in_dim, n_classes, hidden=(256, 128), dropout=0.2):
    layers = []
    last_dim = in_dim
    for h in hidden:
        layers.append(nn.Linear(last_dim, h))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        last_dim = h
    layers.append(nn.Linear(last_dim, n_classes))
    return nn.Sequential(*layers)


def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        if train_mode:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train_mode:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()
    preds_all = []
    true_all = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            preds_all.extend(preds.tolist())
            true_all.extend(yb.numpy().tolist())
    preds_all = np.asarray(preds_all)
    true_all = np.asarray(true_all)
    return preds_all, true_all


### Neural Network Tuning (PyTorch)


In [ ]:
configs = [
    {"hidden": (256, 128), "dropout": 0.2, "lr": 1e-3, "weight_decay": 0.0},
    {"hidden": (512, 256), "dropout": 0.2, "lr": 1e-3, "weight_decay": 0.0},
    {"hidden": (256, 128), "dropout": 0.3, "lr": 5e-4, "weight_decay": 1e-4},
    {"hidden": (512, 256), "dropout": 0.3, "lr": 5e-4, "weight_decay": 1e-4},
]

best_cfg = None
best_val = 0.0
best_state = None
best_model = None

for cfg in configs:
    model = make_mlp(X_tr.shape[1], len(le.classes_), cfg["hidden"], cfg["dropout"]).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_local = 0.0
    best_local_state = None
    epochs = 25
    for _ in range(epochs):
        run_epoch(model, train_loader, criterion, optimizer)
        _, val_acc = run_epoch(model, val_loader, criterion)
        if val_acc > best_local:
            best_local = val_acc
            best_local_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_local > best_val:
        best_val = best_local
        best_cfg = cfg
        best_state = best_local_state
        best_model = model
    print(f"Config {cfg} -> val acc {best_local:.3f}")

if best_state is not None:
    best_model.load_state_dict(best_state)

preds, true = evaluate(best_model, test_loader)
print("Best config:", best_cfg)
print("Test accuracy:", float(np.mean(preds == true)))
print(classification_report(le.inverse_transform(true), le.inverse_transform(preds)))


Config {'hidden': (256, 128), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0} -> val acc 0.574
Config {'hidden': (512, 256), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0} -> val acc 0.556
Config {'hidden': (256, 128), 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001} -> val acc 0.571
Config {'hidden': (512, 256), 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001} -> val acc 0.562
Best config: {'hidden': (256, 128), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0}
Test accuracy: 0.6335869445363326
              precision    recall  f1-score   support

        horn       0.82      0.80      0.81      1319
   left_turn       0.72      0.69      0.71      1327
     neutral       0.64      0.41      0.50      2445
  right_turn       0.87      0.69      0.77      1325
 signal_left       0.59      0.60      0.60      1327
signal_right       0.42      0.81      0.55      1326

    accuracy                           0.63      9069
   macro avg       0.68      0.67      0.65      9